# 00 — Beneficiary Summary Data Profiling

Profiles the raw CMS DE-SynPUF 2008 Beneficiary Summary File (Sample 1) to confirm the assumptions `docs/IMPLEMENTATION_SPEC.md` makes in sections 7-8 before any transformation code is written: chronic-condition flag encoding, death-date null rate, and value ranges for demographics and reimbursement fields.

In [1]:
from pathlib import Path

import pandas as pd

RAW_PATH = Path.cwd().parent / "data" / "raw" / "DE1_0_2008_Beneficiary_Summary_File_Sample_1.csv"

df = pd.read_csv(RAW_PATH, dtype=str)
df.shape

(116352, 32)

## 1. Shape and dtypes

All columns load as strings — typing happens in the silver/staging layer, not here.

In [2]:
df.dtypes

DESYNPUF_ID                 str
BENE_BIRTH_DT               str
BENE_DEATH_DT               str
BENE_SEX_IDENT_CD           str
BENE_RACE_CD                str
BENE_ESRD_IND               str
SP_STATE_CODE               str
BENE_COUNTY_CD              str
BENE_HI_CVRAGE_TOT_MONS     str
BENE_SMI_CVRAGE_TOT_MONS    str
BENE_HMO_CVRAGE_TOT_MONS    str
PLAN_CVRG_MOS_NUM           str
SP_ALZHDMTA                 str
SP_CHF                      str
SP_CHRNKIDN                 str
SP_CNCR                     str
SP_COPD                     str
SP_DEPRESSN                 str
SP_DIABETES                 str
SP_ISCHMCHT                 str
SP_OSTEOPRS                 str
SP_RA_OA                    str
SP_STRKETIA                 str
MEDREIMB_IP                 str
BENRES_IP                   str
PPPYMT_IP                   str
MEDREIMB_OP                 str
BENRES_OP                   str
PPPYMT_OP                   str
MEDREIMB_CAR                str
BENRES_CAR                  str
PPPYMT_C

## 2. Null counts per column

`BENE_DEATH_DT` is expected to be null for the large majority of rows (beneficiaries alive at end of 2008) — confirms spec section 8's rule that blank means `is_deceased = false`, not missing data.

In [3]:
null_counts = df.isna().sum().sort_values(ascending=False)
null_counts[null_counts > 0]

BENE_DEATH_DT    114538
dtype: int64

In [4]:
death_null_rate = df["BENE_DEATH_DT"].isna().mean()
print(f"BENE_DEATH_DT null rate: {death_null_rate:.2%}")

BENE_DEATH_DT null rate: 98.44%


## 3. Demographic code distributions

In [5]:
df["BENE_SEX_IDENT_CD"].value_counts()

BENE_SEX_IDENT_CD
2    64347
1    52005
Name: count, dtype: int64

In [6]:
df["BENE_RACE_CD"].value_counts()

BENE_RACE_CD
1    96349
2    12343
3     4931
5     2729
Name: count, dtype: int64

In [7]:
df["BENE_ESRD_IND"].value_counts()

BENE_ESRD_IND
0    108091
Y      8261
Name: count, dtype: int64

In [8]:
print(f"Distinct SP_STATE_CODE values: {df['SP_STATE_CODE'].nunique()}")
df["SP_STATE_CODE"].value_counts().head(10)

Distinct SP_STATE_CODE values: 52


SP_STATE_CODE
05    10224
10     7745
45     6703
33     6510
39     5199
36     4329
14     4277
23     4012
34     3935
31     3176
Name: count, dtype: int64

## 4. Chronic-condition flag encoding

Confirms every `SP_*` condition column takes only the values `{1, 2}` (CMS codebook: `1 = Yes`, `2 = No`) with no unexpected codes, per spec section 8.

In [9]:
condition_cols = [
    "SP_ALZHDMTA", "SP_CHF", "SP_CHRNKIDN", "SP_CNCR", "SP_COPD",
    "SP_DEPRESSN", "SP_DIABETES", "SP_ISCHMCHT", "SP_OSTEOPRS",
    "SP_RA_OA", "SP_STRKETIA",
]

for col in condition_cols:
    values = sorted(df[col].dropna().unique())
    print(f"{col}: {values}")

SP_ALZHDMTA: ['1', '2']
SP_CHF: ['1', '2']
SP_CHRNKIDN: ['1', '2']
SP_CNCR: ['1', '2']
SP_COPD: ['1', '2']
SP_DEPRESSN: ['1', '2']
SP_DIABETES: ['1', '2']
SP_ISCHMCHT: ['1', '2']
SP_OSTEOPRS: ['1', '2']
SP_RA_OA: ['1', '2']
SP_STRKETIA: ['1', '2']


In [10]:
prevalence = pd.Series(
    {col: (df[col] == "1").mean() for col in condition_cols}
).sort_values(ascending=False)
prevalence.map(lambda x: f"{x:.1%}")

SP_ISCHMCHT    42.1%
SP_DIABETES    37.9%
SP_CHF         28.5%
SP_DEPRESSN    21.3%
SP_ALZHDMTA    19.3%
SP_OSTEOPRS    17.3%
SP_CHRNKIDN    16.1%
SP_RA_OA       15.4%
SP_COPD        13.5%
SP_CNCR         6.4%
SP_STRKETIA     4.5%
dtype: str

In [11]:
chronic_condition_count = (df[condition_cols] == "1").sum(axis=1)
chronic_condition_count.describe()

count    116352.000000
mean          2.222282
std           2.447268
min           0.000000
25%           0.000000
50%           1.000000
75%           4.000000
max          11.000000
dtype: float64

## 5. Coverage-month fields

In [12]:
coverage_cols = [
    "BENE_HI_CVRAGE_TOT_MONS", "BENE_SMI_CVRAGE_TOT_MONS",
    "BENE_HMO_CVRAGE_TOT_MONS", "PLAN_CVRG_MOS_NUM",
]
df[coverage_cols].astype(int).describe()

,BENE_HI_CVRAGE_TOT_MONS,BENE_SMI_CVRAGE_TOT_MONS,BENE_HMO_CVRAGE_TOT_MONS,PLAN_CVRG_MOS_NUM
count,116352.000000,116352.000000,116352.000000,116352.000000
mean,11.143994,10.495514,2.576183,6.826466
std,2.839995,3.758701,4.828831,5.814787
min,0.000000,0.000000,0.000000,0.000000
25%,12.000000,12.000000,0.000000,0.000000
50%,12.000000,12.000000,0.000000,12.000000
75%,12.000000,12.000000,0.000000,12.000000
max,12.000000,12.000000,12.000000,12.000000


## 6. Reimbursement fields

Confirms these are non-negative and checks the scale of inpatient vs. outpatient vs. carrier amounts, informing the `total_cost` definition in spec section 14.

In [13]:
cost_cols = [
    "MEDREIMB_IP", "BENRES_IP", "PPPYMT_IP",
    "MEDREIMB_OP", "BENRES_OP", "PPPYMT_OP",
    "MEDREIMB_CAR", "BENRES_CAR", "PPPYMT_CAR",
]
df[cost_cols] = df[cost_cols].astype(float)
df[cost_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
MEDREIMB_IP,116352.0,2214.180762,8473.340573,-3000.0,0.0,0.0,0.0,164220.0
BENRES_IP,116352.0,249.053441,885.356400,0.0,0.0,0.0,0.0,53096.0
PPPYMT_IP,116352.0,99.142258,1857.930522,0.0,0.0,0.0,0.0,68000.0
MEDREIMB_OP,116352.0,622.226520,1796.476653,-90.0,0.0,20.0,550.0,50020.0
BENRES_OP,116352.0,197.502235,522.437900,0.0,0.0,0.0,180.0,12450.0
PPPYMT_OP,116352.0,25.724182,370.974324,0.0,0.0,0.0,0.0,14400.0
MEDREIMB_CAR,116352.0,1162.095881,1587.643182,0.0,0.0,610.0,1650.0,21160.0
BENRES_CAR,116352.0,328.747508,436.858441,0.0,0.0,170.0,480.0,5260.0
PPPYMT_CAR,116352.0,18.355851,87.356293,0.0,0.0,0.0,0.0,2110.0


**Negative values found** — do not assume reimbursement fields are non-negative. Some rows carry negative `MEDREIMB_IP`/`MEDREIMB_OP` (claim adjustments/reversals are a known feature of CMS reimbursement data). This must be preserved, not filtered out or clipped to zero, by the silver/staging aggregation logic in spec section 14 — treat as a real data characteristic, not a data-quality defect.

In [14]:
for col in cost_cols:
    negative_count = (df[col] < 0).sum()
    if negative_count:
        print(f"{col}: {negative_count} negative rows, min={df[col].min()}")

MEDREIMB_IP: 12 negative rows, min=-3000.0
MEDREIMB_OP: 51 negative rows, min=-90.0


In [15]:
total_cost = df["MEDREIMB_IP"] + df["MEDREIMB_OP"] + df["MEDREIMB_CAR"]
print(f"Beneficiaries with $0 total Medicare-paid cost: {(total_cost == 0).mean():.1%}")
print(f"Beneficiaries with negative total Medicare-paid cost: {(total_cost < 0).sum()}")
total_cost.describe()

Beneficiaries with $0 total Medicare-paid cost: 26.4%
Beneficiaries with negative total Medicare-paid cost: 3


count    116352.000000
mean       3998.503163
std        9847.702381
min        -430.000000
25%           0.000000
50%         940.000000
75%        3040.000000
max      170190.000000
dtype: float64

## 7. Age computation sanity check

Ages beneficiaries as of 2008-12-31 (the file's reference year, per spec section 14) and checks the distribution falls in a plausible Medicare-population range.

In [16]:
birth_date = pd.to_datetime(df["BENE_BIRTH_DT"], format="%Y%m%d")
reference_date = pd.Timestamp("2008-12-31")
age_2008 = (reference_date - birth_date).dt.days // 365
age_2008.describe()

count    116352.000000
mean         71.728797
std          12.507715
min          25.000000
25%          66.000000
50%          72.000000
75%          80.000000
max         100.000000
Name: BENE_BIRTH_DT, dtype: float64

In [17]:
under_65 = (age_2008 < 65).mean()
print(f"Share of beneficiaries under 65 at end of 2008 (ESRD/disability-based enrollees): {under_65:.1%}")

Share of beneficiaries under 65 at end of 2008 (ESRD/disability-based enrollees): 16.4%


## 8. Findings summary

- `BENE_DEATH_DT` is null for the vast majority of rows — treat blank as `is_deceased = false`, confirmed.
- All 11 `SP_*` condition columns take only `{1, 2}` — the `1=Yes`/`2=No` decoding rule in spec section 8 is confirmed against the real file, not assumed.
- Reimbursement fields are **not** guaranteed non-negative — a small number of `MEDREIMB_IP`/`MEDREIMB_OP` rows are negative (claim adjustments), and a non-trivial share of beneficiaries have $0 total Medicare-paid cost in 2008. The silver/staging layer must preserve both, not clip or drop them as errors.
- A meaningful share of beneficiaries are under 65, consistent with ESRD/disability-based Medicare eligibility — the age-band logic in spec section 14 needs an under-65 band, not just the standard 65+ bands.